In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- 1. LOAD APPLE STOCK DATA ---

ticker     = 'AAPL'
start_date = '2020-01-01'
end_date   = '2023-12-31'

raw_df = yf.download(ticker, start=start_date, end=end_date)
raw_df['target'] = raw_df['Close'].shift(-1)
raw_df = raw_df.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume']
X_raw = raw_df[features].values
y_raw = raw_df['target'].values.reshape(-1, 1)

print(f"Loaded {len(raw_df)} trading days: {raw_df.index[0].date()} to {raw_df.index[-1].date()}")

# --- 2. SCALE DATA ---

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

# --- 3. BUILD SLIDING WINDOWS (60-day lookback) ---

SEQ_LEN = 60
X_windows, y_windows = [], []
for i in range(SEQ_LEN, len(X_scaled)):
    X_windows.append(X_scaled[i - SEQ_LEN:i])
    y_windows.append(y_scaled[i])

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)
print(f"Windows: X={X_windows.shape}, y={y_windows.shape}")

# --- 4. CHRONOLOGICAL SPLIT (70 / 15 / 15) ---

n_total = len(X_windows)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)

X_train, y_train = X_windows[:n_train],              y_windows[:n_train]
X_val,   y_val   = X_windows[n_train:n_train+n_val], y_windows[n_train:n_train+n_val]
X_test,  y_test  = X_windows[n_train+n_val:],        y_windows[n_train+n_val:]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# --- 5. PYTORCH DATASET AND DATALOADERS ---

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size   = 64
train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(StockDataset(X_val,   y_val),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(StockDataset(X_test,  y_test),  batch_size=batch_size, shuffle=False)

# --- 6. LSTM MODEL ---

class LSTMPredictor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers,
                               batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 64)
        self.fc2     = nn.Linear(64, 1)
        self.relu    = nn.ReLU()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.dropout(lstm_out[:, -1, :])
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model     = LSTMPredictor(input_size=5).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# --- 7. TRAINING LOOP ---

epochs = 100
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    t_loss = 0.0
    for bX, bY in train_loader:
        bX, bY = bX.to(device), bY.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bX), bY)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for bX, bY in val_loader:
            v_loss += criterion(model(bX.to(device)), bY.to(device)).item()

    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss / len(val_loader))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  train={train_losses[-1]:.6f}  val={val_losses[-1]:.6f}")

# --- 8. EVALUATE ON TEST SET ---

model.eval()
preds, actuals = [], []
with torch.no_grad():
    for bX, bY in test_loader:
        preds.extend(model(bX.to(device)).cpu().numpy())
        actuals.extend(bY.numpy())

preds   = scaler_y.inverse_transform(np.array(preds).reshape(-1, 1))
actuals = scaler_y.inverse_transform(np.array(actuals).reshape(-1, 1))

r2 = r2_score(actuals, preds)
print(f"\nTest R²: {r2:.4f}")

# --- 9. PLOT RESULTS ---

test_dates = raw_df.index[SEQ_LEN + n_train + n_val:]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(test_dates, actuals, label='Actual',    color='blue', alpha=0.7)
axes[0].plot(test_dates, preds,   label='Predicted', color='red',  alpha=0.7)
axes[0].set_title(f'AAPL Stock Price Prediction — LSTM (R²={r2:.4f})')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_losses, label='Train loss')
axes[1].plot(val_losses,   label='Val loss')
axes[1].set_title('Training and Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stock_prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 10. SAVE MODEL ---

torch.save({
    'model_state_dict': model.state_dict(),
    'model_architecture': {'input_size': 5, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2},
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
}, 'lstm_model.pth')
print("Model saved to lstm_model.pth")

## Load model and predict on new data

In [ ]:
def predict_next_day(model_path, new_data):
    """Load a saved LSTM and predict the next closing price."""
    checkpoint = torch.load(model_path)
    model = LSTMPredictor(**checkpoint['model_architecture'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    sx = checkpoint['scaler_X']
    sy = checkpoint['scaler_y']

    if len(new_data) < SEQ_LEN:
        raise ValueError(f"Need at least {SEQ_LEN} data points")

    last_window = sx.transform(new_data[-SEQ_LEN:]).reshape(1, SEQ_LEN, -1)
    with torch.no_grad():
        pred_scaled = model(torch.FloatTensor(last_window))
    return sy.inverse_transform(pred_scaled.numpy())[0][0]


try:
    new_raw = yf.download('AAPL', start='2024-01-01', end='2024-02-01')
    new_X   = new_raw[features].values
    price   = predict_next_day('lstm_model.pth', new_X)
    print(f"Predicted next closing price: ${price:.2f}")
except Exception as e:
    print(f"Prediction failed: {e}")